# Check the MLFlow experiments for the best models

## Import Libraries

In [1]:
import pandas as pd
import mlflow
import re
import os
import shutil
from mlflow.tracking import MlflowClient

## Instantiate MlflowClient and getting experiments name

In [2]:
mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(mlflow_uri)
client = MlflowClient(tracking_uri=mlflow_uri)
experiments = client.search_experiments()
experiments_name = []

for experiment in experiments:
    if experiment.name != "Default":
        experiments_name.append(experiment.name)

In [3]:
runs_df = mlflow.search_runs(experiment_names=experiments_name)

# Display the results
print(f"Retrieved {len(runs_df)} runs!")
display(runs_df.head())

Retrieved 72 runs!


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_recursive_rmse,metrics.best_val_recursive_rmse,params.time_steps,params.train_percentage,...,params.lstm_units,params.horizon_steps,params.batch_size,tags.mlflow.source.git.repoURL,tags.mlflow.source.git.commit,tags.mlflow.source.git.branch,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.user,tags.mlflow.source.name
0,aa5640c261e5429697e32799a1175eb2,6,FINISHED,mlflow-artifacts:/6/aa5640c261e5429697e32799a1...,2026-06-17 18:06:14.689000+00:00,2026-06-17 18:08:11.920000+00:00,0.076793,0.049845,30,0.75,...,32,30,32,https://github.com/rafa-rm/agro-forecaster.git,f05c65257cfa126d4f3b65b5c03960eae675c1ad,main,Run_learning_rate_0.001_cnn_filters_0_lstm_uni...,LOCAL,rafae,.\scripts\tune_architectures.py
1,f6dce246d29b48bba690b383546497d6,6,FINISHED,mlflow-artifacts:/6/f6dce246d29b48bba690b38354...,2026-06-17 18:04:48.346000+00:00,2026-06-17 18:06:14.649000+00:00,0.072951,0.049233,30,0.75,...,32,30,32,https://github.com/rafa-rm/agro-forecaster.git,f05c65257cfa126d4f3b65b5c03960eae675c1ad,main,Run_learning_rate_0.001_cnn_filters_0_lstm_uni...,LOCAL,rafae,.\scripts\tune_architectures.py
2,c879f7b16a254d8596c9b38557e7d2c3,6,FINISHED,mlflow-artifacts:/6/c879f7b16a254d8596c9b38557...,2026-06-17 18:02:49.398000+00:00,2026-06-17 18:04:48.306000+00:00,0.065795,0.049403,30,0.75,...,64,30,32,https://github.com/rafa-rm/agro-forecaster.git,f05c65257cfa126d4f3b65b5c03960eae675c1ad,main,Run_learning_rate_0.001_cnn_filters_0_lstm_uni...,LOCAL,rafae,.\scripts\tune_architectures.py
3,9b815b34165b45aa87a33e011ef87d41,6,FINISHED,mlflow-artifacts:/6/9b815b34165b45aa87a33e011e...,2026-06-17 18:00:12.626000+00:00,2026-06-17 18:02:49.359000+00:00,0.055067,0.050118,30,0.75,...,64,30,32,https://github.com/rafa-rm/agro-forecaster.git,f05c65257cfa126d4f3b65b5c03960eae675c1ad,main,Run_learning_rate_0.001_cnn_filters_0_lstm_uni...,LOCAL,rafae,.\scripts\tune_architectures.py
4,6069f6f811bf45cd9a8d0eb425b80338,6,FINISHED,mlflow-artifacts:/6/6069f6f811bf45cd9a8d0eb425...,2026-06-17 17:59:06.884000+00:00,2026-06-17 18:00:12.585000+00:00,0.073905,0.049042,30,0.75,...,32,30,32,https://github.com/rafa-rm/agro-forecaster.git,f05c65257cfa126d4f3b65b5c03960eae675c1ad,main,Run_learning_rate_0.001_cnn_filters_32_lstm_un...,LOCAL,rafae,.\scripts\tune_architectures.py


## Functions to get the three best models by experiment and format the results

### Format and Retrieval Functions
The following functions format the MLFlow experiment and model names for better readability and consistency. Additionally, they query MLflow to retrieve the top-performing models based on the recursive RMSE metric.


In [4]:
def format_experiment_name(exp_name):
    match = re.search(r'Commodity_([a-zA-Z]+)_TimeSteps_(\d+)', exp_name)
    if match:
        commodity = match.group(1).capitalize() 
        timesteps = match.group(2)
        return f"{commodity}_{timesteps}"
    return exp_name 

def format_model_name(model_name):
    name = model_name.replace("Run_", "")
    name = name.replace("learning_rate_", "lr_")
    name = name.replace("cnn_filters_", "cnn_")
    name = name.replace("lstm_units_", "lstm_")
    name = name.replace("dense_units_", "dense_")
    return name

def get_top_models_per_experiment(experiments, top_k=3):
    target_metric = "metrics.best_val_recursive_rmse"
    run_name_tag = "tags.mlflow.runName"
    
    top_models_list = []

    for exp_name in experiments:
        try:
            runs_df = mlflow.search_runs(
                experiment_names=[exp_name],
            )
            runs_df = runs_df[[run_name_tag, target_metric, "run_id"]]
            runs_df = runs_df.sort_values(by=target_metric, ascending=True)
            runs_df = runs_df.head(top_k)

            for rank_idx, (_, row) in enumerate(runs_df.iterrows(), start=1):
                top_models_list.append({
                    "Experiment": format_experiment_name(exp_name),
                    "Model Name": format_model_name(row[run_name_tag]),
                    "Best Val Recursive RMSE": row[target_metric],
                    "Rank": rank_idx,
                    "Run ID": row["run_id"]
                })
                
        except Exception as e:
            print(f"❌ Error processing {exp_name}: {e}")

    return pd.DataFrame(top_models_list)

## Running the functions

Getting the top three models per experiment and formating the data in a single table to add in github readme

In [5]:
top_models_df = get_top_models_per_experiment(experiments_name)

top_models_df["Model Details"] = top_models_df.apply(
    lambda row: f"{row['Model Name']} (RMSE: {row['Best Val Recursive RMSE']:.4f})", 
    axis=1
)
pivot_df = top_models_df.pivot(
    index="Experiment", 
    columns="Rank", 
    values="Model Details"
).reset_index()

pivot_df.columns.name = None

pivot_df.columns = ["Experiment", "🥇 1st Place", "🥈 2nd Place", "🥉 3rd Place"]

markdown_output = pivot_df.to_markdown(index=False)

print(markdown_output)

| Experiment   | 🥇 1st Place                                   | 🥈 2nd Place                                    | 🥉 3rd Place                                   |
|:-------------|:-----------------------------------------------|:------------------------------------------------|:-----------------------------------------------|
| Corn_30      | lr_0.001_cnn_64_lstm_64_dense_0 (RMSE: 0.0634) | lr_0.001_cnn_32_lstm_64_dense_32 (RMSE: 0.0638) | lr_0.001_cnn_0_lstm_32_dense_32 (RMSE: 0.0640) |
| Corn_7       | lr_0.001_cnn_0_lstm_32_dense_0 (RMSE: 0.0633)  | lr_0.001_cnn_0_lstm_64_dense_0 (RMSE: 0.0636)   | lr_0.001_cnn_32_lstm_32_dense_0 (RMSE: 0.0636) |
| Soy_30       | lr_0.001_cnn_0_lstm_32_dense_32 (RMSE: 0.0602) | lr_0.001_cnn_64_lstm_32_dense_0 (RMSE: 0.0607)  | lr_0.001_cnn_32_lstm_64_dense_0 (RMSE: 0.0608) |
| Soy_7        | lr_0.001_cnn_32_lstm_64_dense_0 (RMSE: 0.0609) | lr_0.001_cnn_32_lstm_32_dense_0 (RMSE: 0.0611)  | lr_0.001_cnn_64_lstm_64_dense_0 (RMSE: 0.0612) |
| Wheat_30   

## Downloading the best model for each experiment and saving in "models/production" folder

### Model Retrieval and Production Deployment
This function reverse-engineers the local folder paths created during training, locates the physical Keras files for the 1st-place models, and securely copies them into the designated production folder (models/production) for deployment.


In [11]:
def organize_local_keras_models_exact(df, source_base="models", target_base="models/production"):
    best_models = df[df["Rank"] == 1]
    
    for _, row in best_models.iterrows():
        exp_clean = row["Experiment"] 
        formatted_model_name = row["Model Name"] 
        
        # 1. Reverse-engineer the Keras folder path
        commodity = exp_clean.split('_')[0] 
        window = exp_clean.split('_')[1]   
        
        # Fixed the string formatting for source_dir to ensure proper slashes
        source_dir = os.path.join(source_base, f"commodity_{commodity.lower()}__window_{window}")
        
        # 2. Map MLflow parameter names back to the training script parameter names
        original_params = formatted_model_name.replace("lr_", "learning_rate_")
        original_params = original_params.replace("cnn_", "cnn_filters_")
        original_params = original_params.replace("lstm_", "lstm_units_")
        original_params = original_params.replace("dense_", "dense_units_")
        
        # 3. Remove the period to match the sanitize_filename() output from training
        original_params = original_params.replace(".", "")
        
        exact_file_name = f"best_model_{original_params}.keras"
        source_file = os.path.join(source_dir, exact_file_name)
        
        target_dir = os.path.join(target_base, exp_clean)
        target_file = os.path.join(target_dir, "best_model.keras")
        os.makedirs(target_dir, exist_ok=True)
        
        print(f"🔍 Looking for exactly: {exact_file_name}")
        
        try:
            if os.path.exists(source_file):
                print(f"⬇️ Copying to {target_file}")
                shutil.copy2(source_file, target_file)
                print("✅ Success!\n")
            else:
                print(f"❌ File not found: {source_file}")
                print(f"   Ensure the folder name matches your target_col exact casing.\n")
                
        except Exception as e:
            print(f"❌ Error processing {exp_clean}: {e}\n")

# Execute
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
source_base = os.path.join(root_path, "models")
organize_local_keras_models_exact(top_models_df, source_base=source_base, target_base=os.path.join(root_path, "models/production"))

🔍 Looking for exactly: best_model_learning_rate_0001_cnn_filters_64_lstm_units_64_dense_units_0.keras
⬇️ Copying to c:\Users\rafae\OneDrive\Documents\GitHub\Agro-Forecaster\research\models/production\Wheat_30\best_model.keras
✅ Success!

🔍 Looking for exactly: best_model_learning_rate_0001_cnn_filters_0_lstm_units_64_dense_units_32.keras
⬇️ Copying to c:\Users\rafae\OneDrive\Documents\GitHub\Agro-Forecaster\research\models/production\Wheat_7\best_model.keras
✅ Success!

🔍 Looking for exactly: best_model_learning_rate_0001_cnn_filters_64_lstm_units_64_dense_units_0.keras
⬇️ Copying to c:\Users\rafae\OneDrive\Documents\GitHub\Agro-Forecaster\research\models/production\Corn_30\best_model.keras
✅ Success!

🔍 Looking for exactly: best_model_learning_rate_0001_cnn_filters_0_lstm_units_32_dense_units_0.keras
⬇️ Copying to c:\Users\rafae\OneDrive\Documents\GitHub\Agro-Forecaster\research\models/production\Corn_7\best_model.keras
✅ Success!

🔍 Looking for exactly: best_model_learning_rate_0001_